#Lib

In [1]:
!pip install ultralytics
!pip install mlflow
!pip install -q dagshub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 128.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 140.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2

In [2]:
from google.colab import drive
import mlflow
import os
from ultralytics import YOLO, settings
import shutil
import dagshub
from google.colab import userdata

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!mkdir /content/datasets
!unzip -o "/content/drive/MyDrive/datn/dot2/experiments/dataset/My First Project.v3i.yolov8.zip" -d "/content/datasets"

Archive:  /content/drive/MyDrive/datn/dot2/experiments/dataset/My First Project.v3i.yolov8.zip
 extracting: /content/datasets/train/images/lao-dong-nang-nhoc-1589447775171403097118-0-27-268-456-crop-15894477819562085522342_webp.rf.ee729fd50cdce0682a2e310aa2825148.jpg  
 extracting: /content/datasets/train/images/586040924_1776564739509330_2942460562199377909_n_jpg.rf.e0959c9ff8c8e302932b2bc56b3c30d3.jpg  
 extracting: /content/datasets/train/images/658151884_4307567006221982_2911182171934244633_n_jpg.rf.fb99fd50ac36438e5ba35699023a425b.jpg  
 extracting: /content/datasets/train/images/7_frame_0032_jpg.rf.2aa44cb02b604d495d9d2b76bfb86933.jpg  
 extracting: /content/datasets/train/images/xay-dung_jpg.rf.656e121319e9655ce2951d3ae564e6e9.jpg  
 extracting: /content/datasets/train/images/3_frame_0017_jpg.rf.b693b9732172284018f371977b568f2f.jpg  
 extracting: /content/datasets/train/images/FBDown-to_AQMEfspph2XW68_mj6RyK_WQ2JKux5MXKlU_chcgMcGtRhvCLld5HcLOftwKeyvcPTaN-rj_Ziihj3JHD9a7NHlFOzdKe

In [5]:
DAGSHUB_USERNAME = "danghoanghai042"
DAGSHUB_REPO     = "ppe_v2"
DAGSHUB_TOKEN    = userdata.get('ppe_v1')

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
os.environ["DAGSHUB_USER_TOKEN"]       = DAGSHUB_TOKEN

TRACKING_URI = f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow"
os.environ["MLFLOW_TRACKING_URI"] = TRACKING_URI
mlflow.set_tracking_uri(TRACKING_URI)

#run

In [6]:
experiment_name = "yolov8m_30e_v2"

In [7]:
if mlflow.active_run():
    mlflow.end_run()

settings.update({'mlflow': True})

dagshub.init(
    repo_owner=DAGSHUB_USERNAME,
    repo_name=DAGSHUB_REPO,
    mlflow=True
)

mlflow.set_experiment("PPE_Detection")

model = YOLO('yolov8m.pt')
results = model.train(
    data='/content/datasets/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.0001,
    name=experiment_name,
    project="PPE_Detection",
    device=0
)

print(f"Đã tự động log toàn bộ lên: https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=10ac6714-994f-4763-ae89-e54cf34c50ca&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=34a310e200b5761a2302b05b99b5c72b9cee976dcb499583e4ffca83f3f40f68




Accessing as danghoanghai042

Initialized MLflow to track repo "danghoanghai042/ppe_v2"

Repository danghoanghai042/ppe_v2 initialized!

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_30e_v2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patien

2026/05/28 02:41:14 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/05/28 02:41:14 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(30694528eeb4425ebc26ff94a2b8bcda) to https://dagshub.com/danghoanghai042/ppe_v2.mlflow
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/runs/detect/PPE_Detection/yolov8m_30e_v2
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/30      6.12G      1.773      1.772      1.502         14        640: 100% ━━━━━━━━━━━━ 132/132 1.7it/s 1:16
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.5it/s 5.4s
                   all        229       2019      0.454       0.43      0.428      0.186

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30      7.27G      1.673      1.277      1.429         14        640: 100% ━━━━━━━━━━━━ 132/132 1.6it/s 1:23
                 Class     Images  Instances      Box(P          R    

In [8]:
from google.colab import runtime
runtime.unassign()